# Deck Archetype Trends

Track curated Deck archetypes, team switching, and field matchups across configurable calendar-day replay snapshots. Run `deck/trend_extract.py` before this notebook.

## Goal

This baseline uses only the project's ordered `ARCHETYPE_RULES`. Every unmatched 60-card Deck is grouped into `Other`; fallback main-Pokemon labels are intentionally disabled.

## Setup and Configuration

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
import yaml


def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        direct = candidate
        nested = candidate / 'imitation_learning'
        if (direct / 'deck' / 'trend.py').is_file():
            return direct
        if (nested / 'deck' / 'trend.py').is_file():
            return nested
    raise FileNotFoundError('Could not locate imitation_learning project root')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from deck.analysis import CardCatalog
from deck.trend import (
    add_trend_archetypes,
    build_daily_metrics,
    build_matchups,
    build_team_flows,
    build_team_modal_archetypes,
    parse_month_day,
    select_snapshot_dates,
)

CONFIG_PATH = PROJECT_ROOT / 'cfg' / 'deck_trend.yaml'
CARD_DATA_PATH = PROJECT_ROOT.parent / 'pokemon_tcg_ai_battle' / 'EN_Card_Data.csv'
config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
extract_cfg = config['extract']
analysis_cfg = config['analysis']

SNAPSHOT_INTERVAL_DAYS = int(analysis_cfg['snapshot_interval_days'])
START_DATE = analysis_cfg.get('start_date')
END_DATE = analysis_cfg.get('end_date')
MIN_SHARE_PERCENT = float(analysis_cfg['min_share_percent'])
if type(analysis_cfg['snapshot_interval_days']) is not int:
    raise ValueError('analysis.snapshot_interval_days must be an integer')
if type(analysis_cfg['exclude_mirror_matches']) is not bool:
    raise ValueError('analysis.exclude_mirror_matches must be true or false')
EXCLUDE_MIRROR_MATCHES = analysis_cfg['exclude_mirror_matches']

if SNAPSHOT_INTERVAL_DAYS < 1:
    raise ValueError('analysis.snapshot_interval_days must be >= 1')
if MIN_SHARE_PERCENT < 0 or MIN_SHARE_PERCENT > 100:
    raise ValueError('analysis.min_share_percent must be in [0, 100]')

data_dir = Path(extract_cfg['output'])
if not data_dir.is_absolute():
    data_dir = PROJECT_ROOT / data_dir
analysis_dir = data_dir / 'analysis'
analysis_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')
print('Config:', CONFIG_PATH)
print('Data:', data_dir)
print('Interval days:', SNAPSHOT_INTERVAL_DAYS)
print('Date bounds:', START_DATE, END_DATE)
print('Minimum displayed share:', MIN_SHARE_PERCENT)

## Load and Validate Extracted Data

In [ ]:
shard_paths = list(data_dir.glob('*.players.csv.gz'))
if not shard_paths:
    raise FileNotFoundError(
        f'No trend shards found in {data_dir}. Run deck/trend_extract.py first.'
    )
shard_paths.sort(key=lambda path: parse_month_day(path.name.split('.players.csv.gz')[0]))
player_rows = pd.concat(
    [pd.read_csv(path, dtype={'date': str, 'episode_id': str}) for path in shard_paths],
    ignore_index=True,
)
required_columns = {
    'date', 'episode_id', 'player', 'team_name', 'opponent_team_name',
    'deck', 'reward', 'result',
}
missing = required_columns - set(player_rows.columns)
if missing:
    raise ValueError(f'Extracted rows are missing columns: {sorted(missing)}')

pair_sizes = player_rows.groupby(['date', 'episode_id']).size()
assert pair_sizes.eq(2).all(), 'Every replay must have exactly two player rows'
assert player_rows.groupby(['date', 'episode_id'])['player'].apply(
    lambda values: set(values.astype(int)) == {0, 1}
).all(), 'Every replay must contain player indices 0 and 1'
assert player_rows['team_name'].astype(str).str.strip().ne('').all()
assert player_rows['deck'].map(lambda value: len(json.loads(value)) == 60).all()

print(f'Shards: {len(shard_paths):,}')
print(f'Games: {len(player_rows) // 2:,}')
print(f'Player rows: {len(player_rows):,}')
print(f'Unique teams: {player_rows.team_name.nunique():,}')

## Select Date Snapshots

In [ ]:
snapshot_dates = select_snapshot_dates(
    player_rows['date'].unique(),
    SNAPSHOT_INTERVAL_DAYS,
    start_date=START_DATE,
    end_date=END_DATE,
)
if not snapshot_dates:
    raise ValueError('No available dates remain after applying the analysis settings')
selected_rows = player_rows[player_rows['date'].isin(snapshot_dates)].copy()
selected_rows['date'] = pd.Categorical(
    selected_rows['date'], categories=snapshot_dates, ordered=True
)
pd.DataFrame({'date': snapshot_dates}).to_csv(
    analysis_dir / 'snapshot_dates.csv', index=False, encoding='utf-8-sig'
)
print('Selected snapshots:', snapshot_dates)
display(selected_rows.groupby('date', observed=True).agg(
    games=('episode_id', lambda values: values.nunique()),
    teams=('team_name', 'nunique'),
).reset_index())

## Classification Coverage

In [ ]:
catalog = CardCatalog.from_csv(CARD_DATA_PATH)
annotated = add_trend_archetypes(selected_rows, catalog)
annotated['date'] = pd.Categorical(
    annotated['date'].astype(str), categories=snapshot_dates, ordered=True
)
coverage = (
    annotated.assign(matched=lambda frame: frame['archetype'].ne('Other'))
    .groupby('date', observed=True)
    .agg(
        player_rows=('archetype', 'size'),
        curated_rows=('matched', 'sum'),
        unique_teams=('team_name', 'nunique'),
    )
    .reset_index()
)
coverage['curated_percent'] = 100 * coverage['curated_rows'] / coverage['player_rows']
coverage['other_percent'] = 100 - coverage['curated_percent']
display(coverage)

## Archetype Usage and Win Rate

In [ ]:
daily_metrics = build_daily_metrics(
    annotated, exclude_mirrors=EXCLUDE_MIRROR_MATCHES
)
daily_metrics['date'] = pd.Categorical(
    daily_metrics['date'].astype(str), categories=snapshot_dates, ordered=True
)
assert np.allclose(
    daily_metrics.groupby('date', observed=True)['share_percent'].sum().to_numpy(),
    100.0,
)
assert daily_metrics['non_mirror_win_rate'].dropna().between(0, 1).all()
visible_archetypes = set(
    daily_metrics.groupby('archetype')['share_percent'].max()
    .loc[lambda values: values >= MIN_SHARE_PERCENT]
    .index
)
visible_metrics = daily_metrics[daily_metrics['archetype'].isin(visible_archetypes)]
daily_metrics.to_csv(
    analysis_dir / 'archetype_daily_metrics.csv', index=False, encoding='utf-8-sig'
)
display(daily_metrics.sort_values(['date', 'uses'], ascending=[True, False]))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 11), sharex=True)
sns.lineplot(
    data=visible_metrics, x='date', y='share_percent', hue='archetype',
    marker='o', ax=axes[0],
)
axes[0].set_title('Archetype Share Across Selected Snapshots')
axes[0].set_ylabel('Share of player rows (%)')
axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
sns.lineplot(
    data=visible_metrics, x='date', y='non_mirror_win_rate',
    hue='archetype', marker='o', ax=axes[1], legend=False,
)
axes[1].axhline(0.5, color='red', linestyle='--', linewidth=1)
axes[1].set_title('Non-Mirror Win Rate Across Selected Snapshots')
axes[1].set_ylabel('Win rate')
axes[1].set_xlabel('Snapshot date')
plt.tight_layout()
plt.show()

## Team Modal Archetypes and Retention

In [ ]:
team_modal = build_team_modal_archetypes(annotated)
team_flows, team_retention = build_team_flows(team_modal, snapshot_dates)
team_modal.to_csv(
    analysis_dir / 'team_modal_archetypes.csv', index=False, encoding='utf-8-sig'
)
team_flows.to_csv(
    analysis_dir / 'team_snapshot_flows.csv', index=False, encoding='utf-8-sig'
)
team_retention.to_csv(
    analysis_dir / 'team_retention.csv', index=False, encoding='utf-8-sig'
)
switched_flows = team_flows[
    team_flows['source_archetype'] != team_flows['target_archetype']
]
inflow = switched_flows.groupby('target_archetype')['teams'].sum().rename('inflow')
outflow = switched_flows.groupby('source_archetype')['teams'].sum().rename('outflow')
flow_balance = pd.concat([inflow, outflow], axis=1).fillna(0).astype(int)
flow_balance['net_flow'] = flow_balance['inflow'] - flow_balance['outflow']
display(team_retention)
display(flow_balance.sort_values('net_flow', ascending=False))
display(team_flows.sort_values('teams', ascending=False).head(30))

## Sankey Flow

In [ ]:
share_lookup = daily_metrics.set_index(['date', 'archetype'])['share_percent'].to_dict()
display_rows = annotated[['date', 'archetype']].copy()
display_rows['display_archetype'] = display_rows.apply(
    lambda row: row['archetype']
    if share_lookup.get((row['date'], row['archetype']), 0) >= MIN_SHARE_PERCENT
    else 'Other',
    axis=1,
)
display_shares = (
    display_rows.groupby(['date', 'display_archetype'], observed=True).size()
    .rename('uses').reset_index()
)
display_shares['share_percent'] = 100 * display_shares['uses'] / (
    display_shares.groupby('date', observed=True)['uses'].transform('sum')
)
display_share_lookup = display_shares.set_index(
    ['date', 'display_archetype']
)[ 'share_percent'].to_dict()
display_modal = team_modal.copy()
display_modal['display_archetype'] = display_modal.apply(
    lambda row: row['modal_archetype']
    if share_lookup.get((row['date'], row['modal_archetype']), 0) >= MIN_SHARE_PERCENT
    else 'Other',
    axis=1,
)
display_modal = display_modal.drop(columns='modal_archetype').rename(
    columns={'display_archetype': 'modal_archetype'}
)
display_flows, _ = build_team_flows(display_modal, snapshot_dates)

node_keys = []
for date_label in snapshot_dates:
    day_labels = sorted(
        display_modal.loc[display_modal['date'] == date_label, 'modal_archetype'].unique()
    )
    node_keys.extend((date_label, archetype) for archetype in day_labels)
node_id = {key: index for index, key in enumerate(node_keys)}
node_labels = [
    f'{date_label} | {archetype} {display_share_lookup.get((date_label, archetype), 0):.0f}%'
    for date_label, archetype in node_keys
]
node_x = [
    0.02 + 0.96 * snapshot_dates.index(date_label) / max(1, len(snapshot_dates) - 1)
    for date_label, _ in node_keys
]
sources, targets, values = [], [], []
for row in display_flows.itertuples(index=False):
    source_key = (row.from_date, row.source_archetype)
    target_key = (row.to_date, row.target_archetype)
    if source_key in node_id and target_key in node_id:
        sources.append(node_id[source_key])
        targets.append(node_id[target_key])
        values.append(int(row.teams))
if sources:
    figure = go.Figure(go.Sankey(
        arrangement='snap',
        node=dict(label=node_labels, x=node_x, pad=8, thickness=14),
        link=dict(source=sources, target=targets, value=values),
    ))
    figure.update_layout(
        height=700,
        title='Top-Band Archetype Evolution (Ribbons = Team Transitions)',
    )
    figure.show()
else:
    print('At least two snapshots with overlapping teams are required for a Sankey chart.')

## Matchup Matrix

In [ ]:
matchups = build_matchups(
    annotated, exclude_mirrors=EXCLUDE_MIRROR_MATCHES
)
assert (matchups['wins'] + matchups['losses'] + matchups['draws']).equals(
    matchups['games']
)
matchups.to_csv(
    analysis_dir / 'matchup_matrix_long.csv', index=False, encoding='utf-8-sig'
)
matrix_archetypes = sorted(
    set(matchups['row_archetype']) & visible_archetypes
)
matrix_rows = matchups[
    matchups['row_archetype'].isin(matrix_archetypes)
    & matchups['column_archetype'].isin(matrix_archetypes)
]
win_matrix = matrix_rows.pivot(
    index='row_archetype', columns='column_archetype', values='win_rate'
).reindex(index=matrix_archetypes, columns=matrix_archetypes)
game_matrix = matrix_rows.pivot(
    index='row_archetype', columns='column_archetype', values='games'
).reindex(index=matrix_archetypes, columns=matrix_archetypes)
annotations = win_matrix.copy().astype(object)
for row_name in matrix_archetypes:
    for column_name in matrix_archetypes:
        rate = win_matrix.loc[row_name, column_name]
        games = game_matrix.loc[row_name, column_name]
        annotations.loc[row_name, column_name] = (
            '' if pd.isna(rate) else f'{rate:.0%}\n(n={int(games):,})'
        )
plt.figure(figsize=(max(10, len(matrix_archetypes) * 0.85), max(8, len(matrix_archetypes) * 0.7)))
sns.heatmap(
    win_matrix, annot=annotations, fmt='', cmap='RdYlGn',
    vmin=0, vmax=1, center=0.5, linewidths=0.5,
)
plt.title('Pooled Archetype Matchups (Row Beats Column)')
plt.xlabel('Opponent archetype')
plt.ylabel('Player archetype')
plt.tight_layout()
plt.show()

## Exported Tables

In [ ]:
exported = sorted(path.name for path in analysis_dir.glob('*.csv'))
print('Analysis output:', analysis_dir)
for name in exported:
    print('-', name)

## Takeaways

After execution, review classification coverage before interpreting trends. A large `Other` share means the curated baseline needs more signature rules. Treat matchup rates with small displayed game counts as exploratory rather than stable estimates.